In [1]:

from langgraph.graph import StateGraph , START,END
from langchain_openai import ChatOpenAI
from dotenv import  load_dotenv
from typing import  TypedDict ,Literal , Annotated
from langchain_core.messages import HumanMessage , BaseMessage 
from langgraph.checkpoint.memory import InMemorySaver


c:\Users\KIIT01\Desktop\LLM\LangGraph\LLM_workflow\venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:

model = ChatOpenAI(
    model="stepfun/step-3.5-flash",
    base_url="https://openrouter.ai/api/v1",
    temperature=0.3,
)

In [3]:
from langgraph.graph import  add_messages
class joke_state(TypedDict):
    messages=Annotated[list[BaseMessage],add_messages]
    topic:str
    gen_joke:str
    exp:str
    

In [4]:
def generate_joke(state:joke_state):
    prompt=f"you need to generate the joke on the basis of given topic:{state['topic']}"
    response=model.invoke(prompt).content
    return {'gen_joke':response}

In [5]:
def generate_expreession(state:joke_state):
    prompt=f"you need to generate the explanation  according topic:{state['topic']} and  joke:{state['gen_joke']}"

    response=model.invoke(prompt).content

    return {'exp':response}
    


In [6]:
#define checkpointer
checkpointer=InMemorySaver()

#define state
graph=StateGraph(joke_state)

#define node
graph.add_node('generate_joke',generate_joke)
graph.add_node('generate_expreession',generate_expreession)

#define edge
graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','generate_expreession')
graph.add_edge('generate_expreession',END)

#compile
workflow=graph.compile(checkpointer=checkpointer)

In [7]:
config2={'configurable':{'thread_id':'1'}}

inital_state={
    'topic':'pasta'
}
final_state=workflow.invoke(inital_state,config=config2)



In [8]:
final_state['topic']

'pasta'

In [9]:
final_state['gen_joke']

'**Why did the pasta go to therapy?**  \nBecause it was feeling a little too *al dente*!  \n\n*(Al dente means "to the tooth" in Italian — pasta cooked to be firm when bitten. The joke plays on the idea of being "too firm" emotionally or rigid, like someone who needs to "loosen up" in therapy.)*'

In [10]:
final_state['exp']

'\n**Explanation of the Joke: "Why did the pasta go to therapy?"**\n\nThis joke works on two levels: a **culinary term** and a **play on emotional psychology**.\n\n1.  **The Culinary Term: "Al Dente"**\n    *   "Al dente" is an Italian phrase meaning "to the tooth." In cooking, it describes pasta (and sometimes rice or vegetables) that is cooked just enough to remain firm and slightly resistant when bitten, rather than soft or mushy. It\'s considered the ideal texture for pasta in Italian cuisine.\n\n2.  **The Wordplay & Punchline:**\n    *   The punchline says the pasta was feeling "*too* al dente."\n    *   This takes the physical property of the pasta (being firm) and **personifies** it, applying it to an emotional state.\n    *   In psychology, someone who is "too rigid," " inflexible," or "hard-headed" might be said to need therapy to "loosen up" or become more emotionally flexible.\n    *   Therefore, the joke suggests the pasta\'s *culinary* firmness is a metaphor for *emotional

In [12]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'pasta', 'gen_joke': '**Why did the pasta go to therapy?**  \nBecause it was feeling a little too *al dente*!  \n\n*(Al dente means "to the tooth" in Italian — pasta cooked to be firm when bitten. The joke plays on the idea of being "too firm" emotionally or rigid, like someone who needs to "loosen up" in therapy.)*', 'exp': '\n**Explanation of the Joke: "Why did the pasta go to therapy?"**\n\nThis joke works on two levels: a **culinary term** and a **play on emotional psychology**.\n\n1.  **The Culinary Term: "Al Dente"**\n    *   "Al dente" is an Italian phrase meaning "to the tooth." In cooking, it describes pasta (and sometimes rice or vegetables) that is cooked just enough to remain firm and slightly resistant when bitten, rather than soft or mushy. It\'s considered the ideal texture for pasta in Italian cuisine.\n\n2.  **The Wordplay & Punchline:**\n    *   The punchline says the pasta was feeling "*too* al dente."\n    *   This takes the physical p

In [49]:
list(workflow.get_state_history(config=config2))

[StateSnapshot(values={'topic': 'pasta', 'gen_joke': 'Why did the spaghetti feel so stressed?\n\nBecause it was told it was *al dente*... and it took it personally!', 'exp': '\nExcellent choice! Here\'s the explanation for that joke, broken down by the two key elements: **pasta** and the **wordplay**.\n\n### **1. The Topic: Pasta (Specifically Spaghetti)**\n*   **What it is:** Pasta is a staple Italian food made from dough (usually wheat flour and water or eggs). It comes in hundreds of shapes (spaghetti, penne, fusilli, etc.).\n*   **Key Cooking Term:** When cooking pasta, the ideal texture is called **"al dente."** This is an Italian phrase meaning **"to the tooth."**\n*   **What "al dente" means:** It describes pasta that is cooked just enough to be tender but still firm and slightly resistant when bitten. It\'s not mushy or soft. Achieving *al dente* is considered a sign of perfectly cooked pasta by chefs and enthusiasts.\n\n### **2. The Joke & The Wordplay**\n> **"Why did the spag

In [14]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'pasta', 'gen_joke': '**Why did the pasta go to therapy?**  \nBecause it was feeling a little too *al dente*!  \n\n*(Al dente means "to the tooth" in Italian — pasta cooked to be firm when bitten. The joke plays on the idea of being "too firm" emotionally or rigid, like someone who needs to "loosen up" in therapy.)*', 'exp': '\n**Explanation of the Joke: "Why did the pasta go to therapy?"**\n\nThis joke works on two levels: a **culinary term** and a **play on emotional psychology**.\n\n1.  **The Culinary Term: "Al Dente"**\n    *   "Al dente" is an Italian phrase meaning "to the tooth." In cooking, it describes pasta (and sometimes rice or vegetables) that is cooked just enough to remain firm and slightly resistant when bitten, rather than soft or mushy. It\'s considered the ideal texture for pasta in Italian cuisine.\n\n2.  **The Wordplay & Punchline:**\n    *   The punchline says the pasta was feeling "*too* al dente."\n    *   This takes the physical p